# 09. Full data: all parcels, Sentinel-2 series and Sentinel-1

## Setup

In [ ]:
from pathlib import Path
import json
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").is_file():
    if root == root.parent:
        raise RuntimeError("Open this notebook inside the project repository")
    root = root.parent
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
if str(root) not in sys.path:
    sys.path.insert(0, str(root))  # experiments package

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from experiments.pilots import climate, clustering, data, earth_engine, evaluation, features, full
from experiments.tuning import tuning
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
AREA = "pilots"


def show_and_save(figure, run, name):
    figure.tight_layout()
    figure.savefig(run / f"{name}.png", dpi=150)
    display(figure)
    plt.close(figure)


def repeated(name, X, rows, ok, seeds=(1, 2, 3), params=evaluation.PARAMS):
    """Mean and SD over random-forest seeds of every evaluation view."""
    summaries, details = [], []
    for seed in seeds:
        summary, holdout = evaluation.evaluate(name, X, rows, ok, params, seed)
        summaries.append({**summary, "seed": seed})
        details.extend({**r, "seed": seed} for r in holdout)
    return summaries, details


def summarize(summaries):
    table = pd.DataFrame(summaries)
    metrics = ["random_split_macro_f1", "spatial_cv_macro_f1", "dataset_holdout_macro_f1", "jecam_macro_f1",
               "jecam_carrot_f1", "jecam_cabbage_f1", "jecam_potato_f1"]
    grouped = table.groupby("feature_set", sort=False)
    out = grouped[metrics].agg(["mean", "std"]).round(3)
    out.insert(0, ("pixels_used", ""), grouped["pixels_used"].first())
    return out

## Inputs

In [ ]:
quality_path = ""  # Optional quality.json from Preprocessing 01; blank uses its verified pointer
pixels_per_parcel = 5
subset_run = ""  # Pilot subset run with the Paoay inputs; blank uses the latest
settings = json.loads((root / "settings.local.json").read_text())
quality_handoff, quality = data.load_quality(root, quality_path or None)
subset_run = Path(subset_run) if subset_run else tuning.latest_run(root, "subset", area=AREA)
print("Quality handoff:", quality_handoff)
print("Paoay inputs from:", subset_run)

## All development and primary JECAM parcels

In [ ]:
run = tuning.new_run(root, "full_data", quality_handoff, area=AREA, pixels_per_parcel=pixels_per_parcel,
                     paoay_inputs=str(subset_run))
regions = data.sample_parcels(quality["regions"], {"train": None, "validation": None, "test": None})
rows, s2_values, s2_valid = data.load_series(regions, quality["pixel_paths"], pixels_per_parcel)
print("Fragments:", len(regions), "| pixels:", len(rows))
display(pd.DataFrame(rows).groupby(["partition", "crop", "source_dataset"]).partition_group.nunique().rename("parcels").reset_index())

## Sentinel-1 half-month backscatter

In [ ]:
ee = earth_engine.initialize(settings["earth_engine_project"])
s1_values = earth_engine.sample_s1(ee, rows, run / "cache", progress=lambda m: print(m, flush=True))
print("Pixels with at least 20 half-months of Sentinel-1:", int((np.isfinite(s1_values[..., 0]).sum(axis=1) >= 20).sum()), "of", len(rows))

## Save

In [ ]:
_, subset_arrays = data.load(subset_run)
data.save(run, rows, {"s2_values": s2_values, "s2_valid": s2_valid, "s1_values": s1_values,
                      **{k: subset_arrays[k] for k in ("paoay_cube", "paoay_s1", "paoay_mask")}})
(run / "paoay.json").write_text((subset_run / "paoay.json").read_text())
print("Run folder:", run)